In [ ]:
# import all libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# load raw data


In [ ]:
# parse separators and add has_metadata flag


### Step 1b: Train/Val/Test Splitting

Stratified, grouped 80/10/10 split
- Stratified on: `has_purchase`, history-length bucket, slate-length bucket
- Grouped on: `query` string (rows sharing a query always land in the same split)
- Done by cutting the data into 10 folds: 8 -> train, 1 -> val, 1 -> test

Input to `split_data`: cleaned slate-level dataframe (one row per search/user).
Required columns:
`slate_id`, `query` (string), `candidate_wid_list`, `candidate_label_list` (arrays of equal length), `history_wid_list` (array)

In [ ]:
# split into train/test/val
SEED = 42

def add_split_helper_cols(df, hist_bins=(0, 29, 78, 188, np.inf), slate_bins=(0, 76, 90, 95, np.inf)):
    # Adds has_purchase, hist_len, slate_len and the combined stratum label
    out = df.copy()
    out["has_purchase"] = out["candidate_label_list"].apply(lambda x: int((np.asarray(x) == 3).any()))
    out["hist_len"] = out["history_wid_list"].apply(len)
    out["slate_len"] = out["candidate_wid_list"].apply(len)
    hist_b = pd.cut(out["hist_len"], bins=list(hist_bins), labels=False, include_lowest=True)
    slate_b = pd.cut(out["slate_len"], bins=list(slate_bins), labels=False, include_lowest=True)
    out["stratum"] = (out["has_purchase"].astype(str) + "_" + hist_b.astype(str) + "_" + slate_b.astype(str))
    return out


def split_data(df, seed=SEED, n_folds=10, n_val_folds=1, n_test_folds=1):
    # Returns a copy of df with a 'split' column in {'train','val','test'}
    # 80/10/10 with the defaults. Rows with the same `query` stay in the same split.
    out = add_split_helper_cols(df)
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)

    fold = np.full(len(out), -1, dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(out, y=out["stratum"], groups=out["query"])):
        fold[idx] = k
    assert (fold >= 0).all()

    split = np.where(fold < n_folds - n_val_folds - n_test_folds, "train",
             np.where(fold < n_folds - n_test_folds, "val", "test"))
    out["split"] = split
    return out


def check_split(df):
    # Prints proportions and sanity checks.
    print("Rows per split:\n", df["split"].value_counts(normalize=True).round(4), "\n")
    summary = df.groupby("split").agg(
        rows=("slate_id", "size"),
        purchase_rate=("has_purchase", "mean"),
        mean_hist_len=("hist_len", "mean"),
        median_hist_len=("hist_len", "median"),
        mean_slate_len=("slate_len", "mean"),
    )
    print(summary.round(4), "\n")
    # no query string may appear in more than one split
    n_splits_per_query = df.groupby("query")["split"].nunique()
    assert (n_splits_per_query == 1).all(), "Leak: a query appears in multiple splits"
    print("OK: every query string is in exactly one split")
    n_eval = df[(df["split"] != "train") & (df["has_purchase"] == 1)].groupby("split").size()
    print("Purchase slates usable for evaluation:\n", n_eval.to_string())


def save_split_ids(df, path="split_ids.parquet"):
    df[["slate_id", "split"]].to_parquet(path, index=False)

In [ ]:
df = split_data(df_clean)
check_split(df)
save_split_ids(df)
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)    # full candidate lists, untouched
test_df  = df[df["split"] == "test"].reset_index(drop=True)   # full candidate lists, untouched

### Step 1c: Negative sampling
`sample_negatives` takes the TRAIN split only.

For every training slate:
- keep ALL items with label 1/2/3 (positives)
- keep up to `n_neg` (defalut 20) randomly chosen label-0 items (negatives)
- slates with no positives go through the same rule (up to `n_neg` label-0 items)

Output is one row per (slate, candidate) with columns:
`slate_id`, `wid`, `label`, `y`, `sample_weight`

`y`=1 only for label 3, weights click 1, cart 5, purchase 10

Call it again every epoch with a new `epoch` number to re-sample negatives.


In [ ]:
WEIGHTS = {0: 1.0, 1: 1.0, 2: 5.0, 3: 10.0}

def sample_negatives(train_df, n_neg=20, epoch=0, seed=SEED, weights=WEIGHTS):
    rng = np.random.default_rng(seed + epoch)
    slate_ids, wids, labels = [], [], []

    for sid, w, l in zip(train_df["slate_id"].values,
                         train_df["candidate_wid_list"].values,
                         train_df["candidate_label_list"].values):
        w = np.asarray(w)
        l = np.asarray(l).astype(int)
        pos_idx = np.flatnonzero(l > 0)
        neg_idx = np.flatnonzero(l == 0)
        if len(neg_idx) > n_neg:
            neg_idx = rng.choice(neg_idx, size=n_neg, replace=False)
        keep = np.concatenate([pos_idx, neg_idx])
        slate_ids.append(np.full(len(keep), sid))
        wids.append(w[keep])
        labels.append(l[keep])

    out = pd.DataFrame({
        "slate_id": np.concatenate(slate_ids),
        "wid": np.concatenate(wids),
        "label": np.concatenate(labels).astype(np.int8),
    })

    out["sample_weight"] = out["label"].map(weights).astype(np.float32)
    out["y"] = (out["label"] == 3).astype(np.int8)

    # shuffle so rows of one slate are not stored positives-first
    return out.sample(frac=1.0, random_state=seed + epoch).reset_index(drop=True)


def check_sampling(train_df, sampled, n_neg=20):
    per_slate = sampled.groupby("slate_id").size()
    print("Rows after sampling:", len(sampled), "| slates:", sampled["slate_id"].nunique(),
          "of", len(train_df))
    print("Items per slate (min/median/max):", per_slate.min(), per_slate.median(), per_slate.max())
    print("Label counts:\n", sampled["label"].value_counts().sort_index().to_string())
    neg_per_slate = sampled[sampled["label"] == 0].groupby("slate_id").size()
    assert neg_per_slate.max() <= n_neg
    print("At most", n_neg, "label-0 items per slate")

In [ ]:
sampled = sample_negatives(train_df, n_neg=20, epoch=0)
check_sampling(train_df, sampled)

# Per-epoch loop for the neural model (Model 2):
# Each epoch sees a different set of about 20 label-0 items per slate
# for epoch in range(num_epochs):
#     sampled = sample_negatives(train_df, n_neg=20, epoch=epoch)
#     ... build features / DataLoader from `sampled`, train one epoch ...

# For Model 1, sample once (epoch=0) and save: sampled.to_parquet("train_sampled.parquet")
# Evaluation: use val_df / test_df candidate lists directly, no sampling.